In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')
print("Kütüphaneler yüklendi ✓")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

Kütüphaneler yüklendi ✓
Pandas: 2.3.3
NumPy: 2.4.1


In [7]:
csv_dir = "../data/csv"

csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]
csv_files.sort()

print(f"Bulunan CSV dosyaları ({len(csv_files)} adet):")
for f in csv_files:
    fpath = os.path.join(csv_dir, f)
    size_mb = os.path.getsize(fpath) / (1024*1024)
    print(f"  {f} ({size_mb:.1f} MB)")

Bulunan CSV dosyaları (8 adet):
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv (73.6 MB)
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv (73.3 MB)
  Friday-WorkingHours-Morning.pcap_ISCX.csv (55.6 MB)
  Monday-WorkingHours.pcap_ISCX.csv (168.7 MB)
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv (79.3 MB)
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv (49.6 MB)
  Tuesday-WorkingHours.pcap_ISCX.csv (128.8 MB)
  Wednesday-workingHours.pcap_ISCX.csv (214.7 MB)


In [8]:
dfs = []

for f in csv_files:
    fpath = os.path.join(csv_dir, f)
    df_temp = pd.read_csv(fpath, encoding='utf-8', low_memory=False)
    # Hangi günden geldiğini takip et
    df_temp['source_file'] = f
    dfs.append(df_temp)
    print(f"✓ {f}: {len(df_temp):,} satır, {len(df_temp.columns)} sütun")

df_raw = pd.concat(dfs, ignore_index=True)
print(f"\nToplam birleşik dataset: {len(df_raw):,} satır, {len(df_raw.columns)} sütun")

✓ Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 satır, 80 sütun
✓ Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 satır, 80 sütun
✓ Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 satır, 80 sütun
✓ Monday-WorkingHours.pcap_ISCX.csv: 529,918 satır, 80 sütun
✓ Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,602 satır, 80 sütun
✓ Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170,366 satır, 80 sütun
✓ Tuesday-WorkingHours.pcap_ISCX.csv: 445,909 satır, 80 sütun
✓ Wednesday-workingHours.pcap_ISCX.csv: 692,703 satır, 80 sütun

Toplam birleşik dataset: 2,830,743 satır, 80 sütun


In [9]:
# CIC-IDS2017'nin sütun isimlerinde başta/sonda boşluk olabiliyor (R7 bulgusu)
df_raw.columns = df_raw.columns.str.strip()

print("Sütunlar:")
print(list(df_raw.columns))

Sütunlar:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE 

In [10]:
print("Sınıf dağılımı (Label):")
label_counts = df_raw['Label'].value_counts()
print(label_counts)
print(f"\nToplam benzersiz sınıf: {df_raw['Label'].nunique()}")

Sınıf dağılımı (Label):
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

Toplam benzersiz sınıf: 15


In [11]:
# Benign strateji: Monday'den tüm benign + diğer günlerden 2000'er
BENIGN_PER_FILE = 2000
benign_chunks = []

for f in csv_files:
    fpath = os.path.join(csv_dir, f)
    df_temp = pd.read_csv(fpath, encoding='utf-8', low_memory=False)
    df_temp.columns = df_temp.columns.str.strip()
    df_temp['source_file'] = f
    
    benign = df_temp[df_temp['Label'] == 'BENIGN']
    
    # Monday için tüm benign satırları al
    if 'Monday' in f:
        sampled = benign
        print(f"✓ {f}: TÜM {len(sampled):,} benign alındı (referans gün)")
    else:
        sampled = benign.sample(n=min(BENIGN_PER_FILE, len(benign)), random_state=42)
        print(f"✓ {f}: {len(benign):,} içinden {len(sampled):,} alındı")
    
    benign_chunks.append(sampled)

df_benign = pd.concat(benign_chunks, ignore_index=True)
print(f"\nToplam benign: {len(df_benign):,} satır")

✓ Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 97,718 içinden 2,000 alındı
✓ Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 127,537 içinden 2,000 alındı
✓ Friday-WorkingHours-Morning.pcap_ISCX.csv: 189,067 içinden 2,000 alındı
✓ Monday-WorkingHours.pcap_ISCX.csv: TÜM 529,918 benign alındı (referans gün)
✓ Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,566 içinden 2,000 alındı
✓ Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 168,186 içinden 2,000 alındı
✓ Tuesday-WorkingHours.pcap_ISCX.csv: 432,074 içinden 2,000 alındı
✓ Wednesday-workingHours.pcap_ISCX.csv: 440,031 içinden 2,000 alındı

Toplam benign: 543,918 satır


In [12]:
# Tüm dosyalardan attack satırlarını al
attack_chunks = []

for f in csv_files:
    fpath = os.path.join(csv_dir, f)
    df_temp = pd.read_csv(fpath, encoding='utf-8', low_memory=False)
    df_temp.columns = df_temp.columns.str.strip()
    df_temp['source_file'] = f
    
    attacks = df_temp[df_temp['Label'] != 'BENIGN']
    if len(attacks) > 0:
        attack_chunks.append(attacks)
        print(f"✓ {f}: {len(attacks):,} attack satırı")
    
    del df_temp, attacks

df_attacks = pd.concat(attack_chunks, ignore_index=True)
print(f"\nToplam attack: {len(df_attacks):,} satır")

print("\nSaldırı türleri:")
print(df_attacks['Label'].value_counts())

# Benign + Attack birleştir
df_combined = pd.concat([df_benign, df_attacks], ignore_index=True)

# NOT: Büyük dataset olduğu için burada shuffle yapmıyoruz.
# Shuffle işlemini model eğitimi sırasında train_test_split ile yapacağız.
print(f"\n--- BİRLEŞİK DATASET ---")
print(f"Toplam satır: {len(df_combined):,}")

print(f"\nSınıf dağılımı:")
print(df_combined['Label'].value_counts())

output_path = "../data/csv/combined_dataset.csv"
df_combined.to_csv(output_path, index=False)

size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"\n✓ Kaydedildi: {output_path}")
print(f"  Boyut: {size_mb:.1f} MB")
print(f"  Satır: {len(df_combined):,}")

✓ Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 128,027 attack satırı
✓ Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 158,930 attack satırı
✓ Friday-WorkingHours-Morning.pcap_ISCX.csv: 1,966 attack satırı
✓ Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 36 attack satırı
✓ Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 2,180 attack satırı
✓ Tuesday-WorkingHours.pcap_ISCX.csv: 13,835 attack satırı
✓ Wednesday-workingHours.pcap_ISCX.csv: 252,672 attack satırı

Toplam attack: 557,646 satır

Saldırı türleri:
Label
DoS Hulk                      231073
PortScan                      158930
DDoS                          128027
DoS GoldenEye                  10293
FTP-Patator                     7938
SSH-Patator                     5897
DoS slowloris                   5796
DoS Slowhttptest                5499
Bot                             1966
Web Attack � Brute Force        1507
Web Attack � XSS                 652
Infiltration                      36
Web At

In [13]:
output_path = "../data/csv/combined_dataset.csv"
df_combined.to_csv(output_path, index=False)

size_mb = os.path.getsize(output_path) / (1024*1024)
print(f"✓ Kaydedildi: {output_path}")
print(f"  Boyut: {size_mb:.1f} MB")
print(f"  Satır: {len(df_combined):,}")

✓ Kaydedildi: ../data/csv/combined_dataset.csv
  Boyut: 415.2 MB
  Satır: 1,101,564
